# TED Europa Tender Scraper

Search for public tenders on [TED Europa](https://ted.europa.eu/) and export the results.

**What this notebook does:**
1. Scrapes tender notices from the TED Search API based on your keywords.
2. Saves a **full raw output** locally (CSV + Excel) with all data columns.
3. **Appends** new results to the Tender Data Master Google Sheet (never overwrites existing data).

**Run naming convention:** `<PROJECT_NAME>_<YEAR_RANGE>_<KEYWORDS>_<YYYYMMDD>` (Singapore time, UTC+08:00).

**Pre-requisite:** Run this in Google Colab. It will prompt you to authenticate for Google Sheets access.

In [1]:
# @title
# --- 0. Mount Google Drive & Set Working Directory ---
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=True)
    except ValueError:
        print("\n[ERROR] Colab Drive mount failed. This is a common Google Colab glitch.")
        print("Please go to the top menu: Runtime -> Restart session, and try again.")
        raise
    print("\n[OK] Google Drive mounted successfully.")
except ImportError:
    print("\n[WARN] Not running in Google Colab. Skipping Drive mount.")

import os
import subprocess
import glob

# Try multiple possible mount points for the project folder.
# Google Colab has used both "Shareddrives" and "Shared drives" (with space)
# at different times, so we check both.
_DRIVE_PROJECT_CANDIDATES = [
    '/content/drive/Shareddrives/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA',
    '/content/drive/SharedDrives/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA',
    '/content/drive/Shared drives/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA',
    '/content/drive/MyDrive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA',
    '/content/drive/My Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA',
]

DRIVE_PROJECT_PATH = None
for _candidate in _DRIVE_PROJECT_CANDIDATES:
    try:
        if os.path.isdir(_candidate):
            DRIVE_PROJECT_PATH = _candidate
            print(f"\n[OK] Found project folder: {DRIVE_PROJECT_PATH}")
            break
    except (PermissionError, OSError):
        continue

# Fallback 1: glob for any top-level shared-drive mount containing the path
if DRIVE_PROJECT_PATH is None and os.path.isdir('/content/drive'):
    for _pattern in [
        '/content/drive/*/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA',
    ]:
        _matches = glob.glob(_pattern)
        if _matches:
            DRIVE_PROJECT_PATH = _matches[0]
            print(f"\n[OK] Found project folder via glob: {DRIVE_PROJECT_PATH}")
            break

# Fallback 2: use `find` to locate ted_colab_scraper.py under /content/drive
if DRIVE_PROJECT_PATH is None and os.path.isdir('/content/drive'):
    print("\n[INFO] Standard shared-drive paths not found. Searching for project folder ...")
    try:
        _result = subprocess.run(
            ['find', '/content/drive', '-maxdepth', '10',
             '-name', 'ted_colab_scraper.py', '-type', 'f'],
            capture_output=True, text=True, timeout=120,
        )
        _found_lines = [l.strip() for l in _result.stdout.strip().split('\n') if l.strip()]
        if _found_lines:
            DRIVE_PROJECT_PATH = os.path.dirname(_found_lines[0])
            print(f"[OK] Found project folder via search: {DRIVE_PROJECT_PATH}")
    except (subprocess.TimeoutExpired, OSError, Exception) as _search_err:
        print(f"[WARN] Search failed: {_search_err}")

if DRIVE_PROJECT_PATH and os.path.isdir(DRIVE_PROJECT_PATH):
    os.chdir(DRIVE_PROJECT_PATH)
    print(f"\n[DIR] Working directory: {os.getcwd()}")
else:
    print(f"\n[ERROR] Project folder not found under /content/drive/.")
    print("Check that the shared drive 'CS Shared Drive' is accessible from your account.")
    print("You may need to open the shared drive link once in Google Drive before it appears here.")

    # Debug: list what IS visible under /content/drive so the user can diagnose
    if os.path.isdir('/content/drive'):
        _top_entries = []
        for _entry in os.listdir('/content/drive'):
            _top_entries.append(_entry)
        print(f"\n[DEBUG] Top-level entries in /content/drive: {_top_entries}")
        # Also list one level deeper for each mount point
        for _entry in _top_entries:
            _sub = os.path.join('/content/drive', _entry)
            if os.path.isdir(_sub):
                try:
                    _children = os.listdir(_sub)[:10]
                    print(f"  /content/drive/{_entry}/  ->  {_children}")
                except (PermissionError, OSError):
                    print(f"  /content/drive/{_entry}/  ->  [PermissionError]")


Mounted at /content/drive

[OK] Google Drive mounted successfully.

[OK] Found project folder: /content/drive/Shareddrives/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA

[DIR] Working directory: /content/drive/Shareddrives/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA


In [2]:
# @title
# --- 1. Install Dependencies & Authenticate ---
!pip install -q gspread google-auth pandas openpyxl requests google-api-python-client deep-translator

try:
    from google.colab import auth
    auth.authenticate_user()
    print("\n[OK] Authenticated with Google.")
except ImportError:
    print("\n[WARN] Not in Colab. Ensure you have local Google credentials configured.")

import sys
import os
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Also ensure DRIVE_PROJECT_PATH is on sys.path if it was resolved in Cell 0
try:
    if DRIVE_PROJECT_PATH and DRIVE_PROJECT_PATH not in sys.path:
        sys.path.insert(0, DRIVE_PROJECT_PATH)
        print(f"[OK] Added project path to Python path: {DRIVE_PROJECT_PATH}")
except NameError:
    pass

# Sanity check: try importing the module right now so the user gets feedback early
try:
    import ted_colab_scraper as _check
    print(f"[OK] ted_colab_scraper module found: {_check.__file__}")
except ModuleNotFoundError:
    print("[WARN] ted_colab_scraper not yet importable.")
    print(f"  sys.path = {sys.path[:5]}...")
    print(f"  cwd = {os.getcwd()}")
    print("  Check that Cell 0 found the project folder successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.5 MB/s eta 0:00:00

[OK] Authenticated with Google.
[OK] ted_colab_scraper module found: /content/drive/Shareddrives/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA/ted_colab_scraper.py


---
## 2. Configure Your Search
Backend execution always uses **advanced expert queries**.
You only provide 4 business inputs, then the notebook builds expert queries for you.
**Helpful Links & Instructions:**
- [TED Advanced Search (Expert Queries)](https://ted.europa.eu/expert-search) - Documentation on how to write expert queries (e.g. `FT~"PCR"`).
### Inputs
1. **Scope**: `ACTIVE`, `LATEST`, or `ALL`
2. **Filtered years**:
   - Enter comma-separated 4-digit years (e.g., `2026, 2027`)
   - **Default**: `2024, 2025` (if left blank)
3. **Filtered countries**:
   - Enter full country names OR region codes, comma-separated.
   - Leave blank for **all countries**.
   - **Example**: `EU7` (pulls 7 major EMEA markets) or `SE, NW`
   - **Aliases**: You can use `UK`, `Great Britain`, or `United Kingdom`.
4. **Keywords**: one or more keyword sets separated by `;` or new lines
5. **CPV codes**: comma-separated (e.g., `33*, 38*`). Default matches `33*` and `38*` categories.
6. **Deduplicate results**: choose `yes` to remove duplicate records by key, or `no` to keep all rows.
   - **How dedup works**: rows are keyed by notice publication number when available; otherwise a fallback key from CPV + Buyer + Country + Year. If duplicates exist, the latest row is kept.
### Country Reference Lookup Table
If you need specific countries or regions, use the exact names/codes from this list. You can type the Sub-region codes (EU7, SE, NW) to automatically select all countries in that group.
| Country | Region | Sub-region Code | TDEUROPA 3-Digit Code |
|---------|--------|-----------------|-----------------------|
| France | EMEA | **EU7** | **FRA** |
| Germany | EMEA | **EU7** | **DEU** |
| Italy | EMEA | **EU7** | **ITA** |
| Spain | EMEA | **EU7** | **ESP** |
| United Kingdom | EMEA | **EU7** | **GBR** |
| Ireland | EMEA | **EU7** | **IRL** |
| Austria | EMEA | **EU7** | **AUT** |
| Russia | EMEA | **SE** | **RUS** |
| Poland | EMEA | **SE** | **POL** |
| Ukraine | EMEA | **SE** | **UKR** |
| Greece | EMEA | **SE** | **GRC** |
| Romania | EMEA | **SE** | **ROU** |
| Czechia | EMEA | **SE** | **CZE** |
| Turkey | EMEA | **SE** | **TUR** |
| Netherlands | EMEA | **NW** | **NLD** |
| Switzerland | EMEA | **NW** | **CHE** |
| Finland | EMEA | **NW** | **FIN** |
| Portugal | EMEA | **NW** | **PRT** |
| Israel | EMEA | **NW** | **ISR** |
| Belgium | EMEA | **NW** | **BEL** |
| Denmark | EMEA |  | **DNK** |
| Bulgaria | EMEA |  | **BGR** |
| Cyprus | EMEA |  | **CYP** |
| Estonia | EMEA |  | **EST** |
| Croatia | EMEA |  | **HRV** |
| Hungary | EMEA |  | **HUN** |
| Iceland | EMEA |  | **ISL** |
| Liechtenstein | EMEA |  | **LIE** |
| Lithuania | EMEA |  | **LTU** |
| Luxembourg | EMEA |  | **LUX** |
| Latvia | EMEA |  | **LVA** |
| Malta | EMEA |  | **MLT** |
| Norway | EMEA |  | **NOR** |
| Slovakia | EMEA |  | **SVK** |
| Slovenia | EMEA |  | **SVN** |
| Sweden | EMEA |  | **SWE** |
*Note: EU7, SE, and NW are predefined shortcut groups. Countries with blank Sub-region Code are still valid and selectable by country name/code.*
*Note: You can mix and match, e.g., `EU7, SE, Norway`.*
### Automatic behavior
- Keywords are treated as exact phrases and converted to TED expert query format (e.g., `FT~"PCR reagent"`).
- Filters are appended automatically:
  - Year range from your **filtered years** input
  - Country filter from your **filtered countries** input
  - CPV filter from your **CPV codes** input
- Generated expert queries are printed before execution so users can verify.
### Output
- Excel only (`.xlsx`): run file + consolidated file
- Saved to the configured output folder path you provide
  - Output folder accepts mounted paths (for example `/content/drive/Shareddrives/...`) and Google Drive folder URLs.
- Run filename format: `<PROJECT_NAME>_<YEAR_RANGE>_<KEYWORDS>_<YYYYMMDD>.xlsx`


In [3]:
# @title 2A) Configure Filters & Output

# ==========================================
# SEARCH CONFIGURATION (SIMPLIFIED)
# ==========================================

# 1) Scope
scope = "ALL" # @param ["ACTIVE", "LATEST", "ALL"]

# 2) Filtered years (comma-separated). Leave blank to use default (2024, 2025).
filter_years = "2024, 2025" # @param {type:"string"}

# 3) Filtered countries (full names, comma-separated). Leave blank for all countries.
# Example using sub-regions: EU7, SE, NW
# Example using mixed names: EU7, Russia, Poland
filter_countries = "Sweden" # @param {type:"string"}

# 4) Keywords (semicolon/newline separated).
keywords = "test" # @param {type:"string"}

# 5) CPV codes (comma-separated, supports * wildcard). Leave blank to use no filter.
cpv_default = "33*, 38*, 504*" # @param {type:"string"}

# 6) PROJECT_NAME token used in run naming and sheet partition labels.
project_name = "TED_EUROPA" # @param {type:"string"}
project_name_token = project_name.strip() or "TED_EUROPA"
raw_worksheet_name = "TED_EUROPA_RAW"
sync_to_master = "yes"
deduplicate_results = "yes" # @param ["yes", "no"]
output_folder = "https://drive.google.com/drive/folders/1aRK0Rb3vPO3OQdyGwTDvvuZCTA7taNBE" # @param {type:"string"}

print(f"[OK] Scope: {scope}")
print(f"[OK] Filtered years input: {filter_years or 'DEFAULT (2024, 2025)'}")
print(f"[OK] Filtered countries input: {filter_countries or 'ALL'}")
print(f"[OK] Keywords input: {keywords}")
print(f"[OK] CPV default: {cpv_default}")
print(f"[OK] Output folder input: {output_folder}")
print(f"[OK] PROJECT_NAME: {project_name_token}")
print(f"[OK] Deduplicate results: {deduplicate_results}")


# Reset helper for non-technical users
import ipywidgets as widgets
from IPython.display import display

_DEFAULT_SETTINGS = {
    "scope": "ALL",
    "filter_years": "2024, 2025",
    "filter_countries": "Finland",
    "keywords": "reagent",
    "cpv_default": "33*, 38*",
    "deduplicate_results": "no",
    "project_name": "TED_EUROPA",
    "output_folder": "/content/drive/Shareddrives/New EIU Gateway/0. Gateway Ops & Databases/Tender Data Master/ted_europa_test",
}

def _reset_all_settings(_=None):
    global scope, filter_years, filter_countries, keywords, cpv_default, deduplicate_results, project_name, project_name_token, output_folder
    scope = _DEFAULT_SETTINGS["scope"]
    filter_years = _DEFAULT_SETTINGS["filter_years"]
    filter_countries = _DEFAULT_SETTINGS["filter_countries"]
    keywords = _DEFAULT_SETTINGS["keywords"]
    cpv_default = _DEFAULT_SETTINGS["cpv_default"]
    deduplicate_results = _DEFAULT_SETTINGS["deduplicate_results"]
    project_name = _DEFAULT_SETTINGS["project_name"]
    project_name_token = project_name.strip() or "TED_EUROPA"
    output_folder = _DEFAULT_SETTINGS["output_folder"]
    print("[RESET] Configuration reset to defaults for this runtime.")
    print("[RESET] Re-run this cell, then run Cells 2B and 3.")

reset_button = widgets.Button(description="Reset all settings to default", button_style="warning", icon="refresh")
reset_button.on_click(_reset_all_settings)
display(reset_button)


[OK] Scope: ALL
[OK] Filtered years input: 2024, 2025
[OK] Filtered countries input: Sweden
[OK] Keywords input: test
[OK] CPV default: 33*, 38*, 504*
[OK] Output folder input: https://drive.google.com/drive/folders/1aRK0Rb3vPO3OQdyGwTDvvuZCTA7taNBE
[OK] PROJECT_NAME: TED_EUROPA
[OK] Deduplicate results: yes


Button(button_style='warning', description='Reset all settings to default', icon='refresh', style=ButtonStyle(…

In [4]:
# @title 2B) Build & Verify Expert Queries

# --- 2B. Build & Verify Expert Queries ---
# This step generates the expert queries based on your filters.
# Copy the printed queries below to the next cell to edit them.

from datetime import datetime
import re
import shlex
import importlib
import sys
import os

def _ensure_ted_module_importable():
    """Locate ted_colab_scraper.py and ensure it is importable."""
    import sys, os, glob

    # Quick check: already importable?
    try:
        import ted_colab_scraper as _probe
        return
    except ModuleNotFoundError:
        pass

    # Strategy 1: check if DRIVE_PROJECT_PATH was set by Cell 0
    try:
        _drv = DRIVE_PROJECT_PATH  # noqa: F821
        if _drv and os.path.isfile(os.path.join(_drv, 'ted_colab_scraper.py')):
            if _drv not in sys.path:
                sys.path.insert(0, _drv)
            os.chdir(_drv)
            print(f"[OK] Module path resolved via DRIVE_PROJECT_PATH: {_drv}")
            return
    except NameError:
        pass

    # Strategy 2: check cwd – maybe Cell 0 already cd'd there
    if os.path.isfile(os.path.join(os.getcwd(), 'ted_colab_scraper.py')):
        if os.getcwd() not in sys.path:
            sys.path.insert(0, os.getcwd())
        print(f"[OK] Module found in current directory: {os.getcwd()}")
        return

    # Strategy 3: glob for any shared-drive mount containing the file
    if os.path.isdir('/content/drive'):
        for _pattern in [
            '/content/drive/*/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA/ted_colab_scraper.py',
            '/content/drive/*/*/Data Infrastructure/Tender Data/Scripts/TED_EUROPA/ted_colab_scraper.py',
        ]:
            _matches = glob.glob(_pattern)
            if _matches:
                _module_dir = os.path.dirname(_matches[0])
                if _module_dir not in sys.path:
                    sys.path.insert(0, _module_dir)
                os.chdir(_module_dir)
                print(f"[OK] Found module via glob: {_matches[0]}")
                return

    # Strategy 4: search /content/drive using `find`
    if os.path.isdir('/content/drive'):
        import subprocess
        try:
            _res = subprocess.run(
                ['find', '/content/drive', '-maxdepth', '10',
                 '-name', 'ted_colab_scraper.py', '-type', 'f'],
                capture_output=True, text=True, timeout=120,
            )
            _hits = [l.strip() for l in _res.stdout.strip().split('\n') if l.strip()]
            if _hits:
                _module_dir = os.path.dirname(_hits[0])
                if _module_dir not in sys.path:
                    sys.path.insert(0, _module_dir)
                os.chdir(_module_dir)
                print(f"[OK] Found module via search: {_hits[0]}")
                print(f"[DIR] Working directory set to: {_module_dir}")
                return
        except Exception as _e:
            print(f"[WARN] find search failed: {_e}")

    # Give the user a helpful diagnostic before raising
    _diag = []
    _diag.append(f"  cwd: {os.getcwd()}")
    _diag.append(f"  sys.path (first 5): {sys.path[:5]}")
    try:
        _drv_val = DRIVE_PROJECT_PATH  # noqa: F821
        _diag.append(f"  DRIVE_PROJECT_PATH: {_drv_val}")
    except NameError:
        _diag.append("  DRIVE_PROJECT_PATH: <not set>")
    if os.path.isdir('/content/drive'):
        _top = os.listdir('/content/drive')
        _diag.append(f"  /content/drive contents: {_top}")

    raise ModuleNotFoundError(
        "Could not find 'ted_colab_scraper.py' anywhere under /content/drive.\n"
        "Diagnostics:\n" + "\n".join(_diag) + "\n\n"
        "Make sure:\n"
        "  1. Google Drive is mounted (run Cell 0 first).\n"
        "  2. The shared drive 'CS Shared Drive' is accessible from your Google account.\n"
        "     (Open the shared drive link once in Google Drive to add it.)\n"
        "  3. Re-run Cell 0, then Cell 1, and then try this cell again."
    )

_ensure_ted_module_importable()

import ted_colab_scraper

ted_colab_scraper = importlib.reload(ted_colab_scraper)
from ted_colab_scraper import (
    build_expert_query_from_keywords,
    build_filtered_query,
    parse_country_filter,
    parse_year_filter,
    parse_cpv_filter
)

default_filter_years = [2024, 2025]
filter_year_values = parse_year_filter(filter_years, default_years=default_filter_years)
filter_country_codes, unknown_countries = parse_country_filter(filter_countries)
cpv_values = parse_cpv_filter(cpv_default, default_cpv_codes=['33*', '38*'])

if unknown_countries:
    print(f"[WARN] Unknown country names skipped: {', '.join(unknown_countries)}")

translation_skip_before_year_value = min(filter_year_values)

print(f"\n[FILTERED SCOPE]")
print(f"  Years: {min(filter_year_values)}-{max(filter_year_values)}")
print(f"  Countries: {', '.join(filter_country_codes) if filter_country_codes else 'ALL'}")
print(f"  CPV: {', '.join(cpv_values)}")

entries = [x.strip() for x in re.split(r"[;\n]+", keywords) if x.strip()]
if not entries:
    raise ValueError("[ERROR] Provide at least one keyword set.")

generated_queries = []
for entry in entries:
    tokens = [entry]
    if not tokens:
        continue
    base_query = build_expert_query_from_keywords(tokens)
    expert_query = build_filtered_query(
        base_query,
        years=filter_year_values,
        country_codes=filter_country_codes,
        cpv_codes=cpv_values,
    )
    generated_queries.append({
        "label": entry,
        "expert_query": expert_query
    })

print("\n==========================================")
print("GENERATED EXPERT QUERIES")
print("Copy the queries below into the next cell (Step 2C).")
print("You can edit them there or add your own, separated by commas (,).")
print("==========================================\n")

combined_generated = ",\n\n".join([job['expert_query'] for job in generated_queries])
print(combined_generated)



[FILTERED SCOPE]
  Years: 2024-2025
  Countries: SWE
  CPV: 33*, 38*, 504*

GENERATED EXPERT QUERIES
Copy the queries below into the next cell (Step 2C).
You can edit them there or add your own, separated by commas (,).

(FT~"test") AND buyer-country IN (SWE) AND publication-date = (20240101 <> 20251231) AND (classification-cpv = 33* OR classification-cpv = 38* OR classification-cpv = 504*)


In [5]:
# (Placeholder – no additional installs needed.
#  ted_colab_scraper is a local Python module loaded from Google Drive,
#  not a system package.)
print("[OK] No additional system packages needed.")


[OK] No additional system packages needed.


### 2C. Edit / Add Expert Queries
**Instructions:**
1. Copy the generated expert queries printed in the cell above.
2. Paste them into the `final_expert_queries` text box below.
3. You can modify them or add your own additional queries here.
4. **Make sure multiple queries are separated by a COMMA (`,`).**

In [6]:
final_expert_queries = "" # @param {type:"string"}

In [7]:
# @title 3) Run Scraper

# --- 3. Run Scraper (Uses the query_jobs defined above) ---

from pathlib import Path
from datetime import datetime, timedelta, timezone
import pandas as pd
import os
import re
import sys
import time
from urllib.parse import parse_qs, urlparse

import importlib

def _ensure_ted_module_importable():
    """Locate ted_colab_scraper.py and ensure it is importable."""
    import sys, os, glob

    # Quick check: already importable?
    try:
        import ted_colab_scraper as _probe
        return
    except ModuleNotFoundError:
        pass

    # Strategy 1: check if DRIVE_PROJECT_PATH was set by Cell 0
    try:
        _drv = DRIVE_PROJECT_PATH  # noqa: F821
        if _drv and os.path.isfile(os.path.join(_drv, 'ted_colab_scraper.py')):
            if _drv not in sys.path:
                sys.path.insert(0, _drv)
            os.chdir(_drv)
            print(f"[OK] Module path resolved via DRIVE_PROJECT_PATH: {_drv}")
            return
    except NameError:
        pass

    # Strategy 2: check cwd – maybe Cell 0 already cd'd there
    if os.path.isfile(os.path.join(os.getcwd(), 'ted_colab_scraper.py')):
        if os.getcwd() not in sys.path:
            sys.path.insert(0, os.getcwd())
        print(f"[OK] Module found in current directory: {os.getcwd()}")
        return

    # Strategy 3: glob for any shared-drive mount containing the file
    if os.path.isdir('/content/drive'):
        for _pattern in [
            '/content/drive/*/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA/ted_colab_scraper.py',
            '/content/drive/*/*/Data Infrastructure/Tender Data/Scripts/TED_EUROPA/ted_colab_scraper.py',
        ]:
            _matches = glob.glob(_pattern)
            if _matches:
                _module_dir = os.path.dirname(_matches[0])
                if _module_dir not in sys.path:
                    sys.path.insert(0, _module_dir)
                os.chdir(_module_dir)
                print(f"[OK] Found module via glob: {_matches[0]}")
                return

    # Strategy 4: search /content/drive using `find`
    if os.path.isdir('/content/drive'):
        import subprocess
        try:
            _res = subprocess.run(
                ['find', '/content/drive', '-maxdepth', '10',
                 '-name', 'ted_colab_scraper.py', '-type', 'f'],
                capture_output=True, text=True, timeout=120,
            )
            _hits = [l.strip() for l in _res.stdout.strip().split('\n') if l.strip()]
            if _hits:
                _module_dir = os.path.dirname(_hits[0])
                if _module_dir not in sys.path:
                    sys.path.insert(0, _module_dir)
                os.chdir(_module_dir)
                print(f"[OK] Found module via search: {_hits[0]}")
                print(f"[DIR] Working directory set to: {_module_dir}")
                return
        except Exception as _e:
            print(f"[WARN] find search failed: {_e}")

    # Give the user a helpful diagnostic before raising
    _diag = []
    _diag.append(f"  cwd: {os.getcwd()}")
    _diag.append(f"  sys.path (first 5): {sys.path[:5]}")
    try:
        _drv_val = DRIVE_PROJECT_PATH  # noqa: F821
        _diag.append(f"  DRIVE_PROJECT_PATH: {_drv_val}")
    except NameError:
        _diag.append("  DRIVE_PROJECT_PATH: <not set>")
    if os.path.isdir('/content/drive'):
        _top = os.listdir('/content/drive')
        _diag.append(f"  /content/drive contents: {_top}")

    raise ModuleNotFoundError(
        "Could not find 'ted_colab_scraper.py' anywhere under /content/drive.\n"
        "Diagnostics:\n" + "\n".join(_diag) + "\n\n"
        "Make sure:\n"
        "  1. Google Drive is mounted (run Cell 0 first).\n"
        "  2. The shared drive 'CS Shared Drive' is accessible from your Google account.\n"
        "     (Open the shared drive link once in Google Drive to add it.)\n"
        "  3. Re-run Cell 0, then Cell 1, and then try this cell again."
    )

_ensure_ted_module_importable()

import ted_colab_scraper as _tcs

_tcs = importlib.reload(_tcs)
run_scrape = _tcs.run_scrape
deduplicate_output_rows = _tcs.deduplicate_output_rows
save_run_extract = _tcs.save_run_extract
append_to_consolidated = _tcs.append_to_consolidated
build_run_output_stem = getattr(_tcs, "build_run_output_stem", None)
if build_run_output_stem is None:
    raise ImportError(
        "build_run_output_stem is not available in ted_colab_scraper.py. "
        "Please re-run Cell 0 and Cell 1, then run this cell again."
    )

def _default_output_path() -> Path:
    # Colab default when Drive is mounted; otherwise a local notebook folder.
    try:
        if Path("/content/drive").exists():
            return Path("/content/drive/Shareddrives/New EIU Gateway/0. Gateway Ops & Databases/Tender Data Master/ted_europa")
    except (PermissionError, OSError):
        pass
    return Path.cwd() / "ted_europa_output"

def _local_fallback_output_path() -> Path:
    fallback = Path.cwd() / "ted_europa_output"
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback

DEFAULT_OUTPUT_PATH = _default_output_path()

def _extract_drive_folder_id(raw_value: str) -> str:
    text = str(raw_value or "").strip()
    if not text:
        return ""
    patterns = [
        r"https?://drive\.google\.com/drive/folders/([A-Za-z0-9_-]+)",
        r"https?://drive\.google\.com/open\?id=([A-Za-z0-9_-]+)",
        r"https?://drive\.google\.com/.+[?&]id=([A-Za-z0-9_-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    parsed = urlparse(text)
    if "drive.google.com" in parsed.netloc:
        query = parse_qs(parsed.query)
        return query.get("id", [""])[0]
    return ""


def _resolve_output_target(output_value: str):
    raw_value = str(output_value or "").strip()
    if not raw_value:
        is_drive = str(DEFAULT_OUTPUT_PATH).startswith("/content/drive/")
        return DEFAULT_OUTPUT_PATH, "", "", is_drive

    folder_id = _extract_drive_folder_id(raw_value)
    folder_url = f"https://drive.google.com/drive/folders/{folder_id}" if folder_id else ""

    if raw_value.startswith("/content/drive/"):
        if Path("/content/drive").exists():
            return Path(raw_value), folder_id, folder_url, True
        print("[WARN] /content/drive path provided outside Colab. Using local output folder instead.")
        return _local_fallback_output_path(), folder_id, folder_url, False

    if raw_value.upper().startswith("G:\\"):
        if Path("/content/drive").exists():
            normalized = raw_value.replace("\\", "/")
            shared_prefix = "G:/Shared drives/"
            my_prefix = "G:/My Drive/"
            if normalized.startswith(shared_prefix):
                tail = normalized[len(shared_prefix):]
                return Path("/content/drive/Shareddrives") / tail, folder_id, folder_url, True
            if normalized.startswith(my_prefix):
                tail = normalized[len(my_prefix):]
                return Path("/content/drive/MyDrive") / tail, folder_id, folder_url, True
        print("[WARN] Windows G: path detected outside Colab. Using local output folder instead.")
        return _local_fallback_output_path(), folder_id, folder_url, False

    if folder_id:
        candidate_paths = [
            Path("/content/drive/.shortcut-targets-by-id") / folder_id,
            Path("/content/drive/MyDrive/.shortcut-targets-by-id") / folder_id,
            Path("/content/drive/Shareddrives/.shortcut-targets-by-id") / folder_id,
        ]
        for candidate in candidate_paths:
            try:
                if candidate.exists():
                    return candidate, folder_id, folder_url, True
            except (PermissionError, OSError):
                continue

        fallback_path = _local_fallback_output_path()
        print(
            "[WARN] Could not map the Drive folder URL to a mounted shortcut path. "
            f"Using local output folder instead: {fallback_path}"
        )
        return fallback_path, folder_id, folder_url, False

    return Path(raw_value), "", "", False


OUTPUT_PATH, OUTPUT_FOLDER_ID, OUTPUT_FOLDER_URL, OUTPUT_PATH_IS_DRIVE = _resolve_output_target(output_folder)
try:
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
except OSError as create_error:
    fallback_path = _local_fallback_output_path()
    print(
        f"[WARN] Could not create requested output path ({OUTPUT_PATH}): {create_error}. "
        f"Using local folder instead: {fallback_path}"
    )
    OUTPUT_PATH = fallback_path
    OUTPUT_PATH_IS_DRIVE = False

if not OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"[ERROR] Output folder not accessible: {OUTPUT_PATH}\n"
        "Try a local path, mounted path under /content/drive/..., or a valid Google Drive folder URL."
    )

print(f"[DIR] Scripts folder: {os.getcwd()}")
print(f"[DIR] Output folder:  {OUTPUT_PATH}")
if OUTPUT_FOLDER_URL:
    print(f"[DIR] Output folder URL: {OUTPUT_FOLDER_URL}")
if OUTPUT_FOLDER_ID:
    print(f"[DIR] Target Drive folder ID: {OUTPUT_FOLDER_ID}")

if 'final_expert_queries' not in locals() or not final_expert_queries.strip():
    if 'combined_generated' in locals() and combined_generated.strip():
        print("\n[INFO] You did not paste any custom queries in Cell 2C.")
        print("[INFO] Falling back to the automatically generated queries from Cell 2B.")
        final_queries_text = combined_generated.strip()
    else:
        raise ValueError("[ERROR] Please run Cell 2B first to generate queries, or paste your queries into Cell 2C.")
else:
    final_queries_text = final_expert_queries.strip()
raw_queries = [q.strip() for q in final_queries_text.split(",") if q.strip()]

if not raw_queries:
    raise ValueError("[ERROR] No valid queries found. Please check Cell 2C.")

query_jobs = []
for i, q in enumerate(raw_queries, start=1):
    query_jobs.append({
        "label": f"Custom_{i}",
        "expert_query": q
    })

# Date-only run token for naming convention consistency with the Python script.
run_stamp = datetime.now(timezone(timedelta(hours=8), name="Asia/Singapore")).strftime("%Y%m%d")
dedupe_enabled = str(deduplicate_results).strip().lower() in {"yes", "true", "1", "y"}

ALL_QUERY_TEXT = " || ".join(job["expert_query"] for job in query_jobs)
keyword_tokens = [token.strip() for token in re.split(r"[;,\n]+", keywords or "") if token.strip()]
# Build canonical filename stem: PROJECT + YEAR_RANGE + KEYWORDS + YYYYMMDD.
run_stem = build_run_output_stem(
    project_name=project_name_token,
    years=[str(y) for y in filter_year_values] if filter_year_values else None,
    keywords=keyword_tokens or None,
    expert_query=ALL_QUERY_TEXT,
    run_timestamp=run_stamp,
)
run_filename = f"{run_stem}.xlsx"
print("[NAME] Run identity follows: PROJECT_NAME + YEAR_RANGE + KEYWORDS + DATE_YYYYMMDD")


def run_query_jobs(jobs, stamp):
    frames = []
    query_texts = []
    for idx, job in enumerate(jobs, start=1):
        print(f"\n[RUN] Query {idx}/{len(jobs)}: {job['label']}\n{job['expert_query']}")
        output_prefix = OUTPUT_PATH / f"TED_tmp_{stamp}_{idx:02d}"

        df_part = run_scrape(
            keywords=None,
            expert_query=job["expert_query"],
            output_prefix=output_prefix,
            checkpoint_path=None,
            limit=100,
            scope=scope,
            max_pages=None,
            max_rows=2000,
            pause_seconds=1.25,
            pause_every_pages=5,
            auto_pause_seconds=10.0,
            only_latest_versions=True,
            resume_from_checkpoint=False,
            verbose_progress=True,
            translation_skip_before_year=translation_skip_before_year_value,
            deduplicate_output=dedupe_enabled,
            run_filename=run_filename,
        )

        frames.append(df_part)
        query_texts.append(job["expert_query"])

        tmp_xlsx = output_prefix.with_suffix(".xlsx")
        if tmp_xlsx.exists():
            tmp_xlsx.unlink()
    return frames, query_texts


raw_frames, executed_query_texts = run_query_jobs(query_jobs, run_stamp)
combined_raw = pd.concat(raw_frames, ignore_index=True) if raw_frames else pd.DataFrame()

if combined_raw.empty:
    print("\n[WARN] No data returned after default query attempts. Continuing gracefully.")

if dedupe_enabled:
    df_raw_internal = deduplicate_output_rows(combined_raw)
else:
    df_raw_internal = combined_raw.reset_index(drop=True)
ALL_QUERY_TEXT = " || ".join(executed_query_texts)

print(f"\n[OK] Combined rows before dedup: {len(combined_raw)}")
dedup_removed = max(len(combined_raw) - len(df_raw_internal), 0)
if dedupe_enabled:
    print(f"[OK] Rows after dedup: {len(df_raw_internal)}")
    print("[INFO] Dedup logic: keep the latest row per record key.")
    print("[INFO] Record key source: notice publication number when present; fallback is CPV + Buyer + Country + Year.")
    print(f"[INFO] Duplicate rows removed: {dedup_removed}")
else:
    print(f"[OK] Deduplication disabled. Rows kept: {len(df_raw_internal)}")
    print("[INFO] Dedup disabled: all rows are kept, including repeated record keys.")

runs_dir = OUTPUT_PATH / "runs"
df_raw, run_path = save_run_extract(
    df_raw_internal,
    source="TED Europa API",
    project=project_name_token,
    query_text=ALL_QUERY_TEXT,
    runs_dir=runs_dir,
    run_timestamp=run_stamp,
    run_filename=run_filename,
)
print(f"[SAVE] Run extract saved to: {run_path}")

consolidated_prefix = OUTPUT_PATH / "ted_consolidated_raw"
append_to_consolidated(df_raw, consolidated_prefix=consolidated_prefix)

RAW_OUTPUT_URL = str(run_path)
if OUTPUT_FOLDER_ID:
    try:
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload
        from google.auth import default as _auth_default
        import httplib2
        import google_auth_httplib2

        creds, _ = _auth_default()
        _drive_http = httplib2.Http(timeout=60)
        _authed_http = google_auth_httplib2.AuthorizedHttp(creds, http=_drive_http)
        drive_service = build('drive', 'v3', http=_authed_http, cache_discovery=False)

        def _lookup_drive_file(file_name: str, attempts: int = 5, delay_seconds: float = 2.0):
            query_name = file_name.replace("'", "\\'")
            query = f"name = '{query_name}' and '{OUTPUT_FOLDER_ID}' in parents and trashed = false"
            for _ in range(attempts):
                response = drive_service.files().list(
                    q=query,
                    spaces='drive',
                    fields='files(id, webViewLink)',
                    includeItemsFromAllDrives=True,
                    supportsAllDrives=True,
                    pageSize=1,
                ).execute()
                files = response.get('files', [])
                if files:
                    return files[0]
                time.sleep(delay_seconds)
            return None

        def _upload_to_folder(local_path: str, file_name: str, mime_type: str):
            metadata = {'name': file_name, 'parents': [OUTPUT_FOLDER_ID]}
            media = MediaFileUpload(local_path, mimetype=mime_type, resumable=False)
            created = drive_service.files().create(
                body=metadata,
                media_body=media,
                fields='id, webViewLink',
                supportsAllDrives=True,
            ).execute()
            return created

        drive_file_meta = _lookup_drive_file(run_filename)
        if drive_file_meta is None:
            created_xlsx = _upload_to_folder(
                str(run_path),
                run_filename,
                'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
            )
            RAW_OUTPUT_URL = created_xlsx.get('webViewLink') or f"https://drive.google.com/file/d/{created_xlsx['id']}/view"
            print("[SAVE] Uploaded XLSX to target Google Drive folder URL.")
        else:
            RAW_OUTPUT_URL = drive_file_meta.get('webViewLink') or f"https://drive.google.com/file/d/{drive_file_meta['id']}/view"
            print("[SAVE] Located XLSX in target Google Drive folder.")
    except Exception as upload_error:
        print(f"[WARN] Could not validate/upload Drive output URL: {upload_error}")

print(f"[SAVE] Output directory path: {OUTPUT_PATH}")
if OUTPUT_FOLDER_URL:
    print(f"[SAVE] Output directory URL:  {OUTPUT_FOLDER_URL}")
print(f"[SAVE] Raw output URL for sync: {RAW_OUTPUT_URL}")
print(f"[SAVE] Files successfully generated in: {OUTPUT_PATH}")


[DIR] Scripts folder: /content/drive/Shareddrives/CS Shared Drive/Data Infrastructure/Tender Data/Scripts/TED_EUROPA
[DIR] Output folder:  /content/drive/.shortcut-targets-by-id/1aRK0Rb3vPO3OQdyGwTDvvuZCTA7taNBE
[DIR] Output folder URL: https://drive.google.com/drive/folders/1aRK0Rb3vPO3OQdyGwTDvvuZCTA7taNBE
[DIR] Target Drive folder ID: 1aRK0Rb3vPO3OQdyGwTDvvuZCTA7taNBE

[INFO] You did not paste any custom queries in Cell 2C.
[INFO] Falling back to the automatically generated queries from Cell 2B.
[NAME] Run identity follows: PROJECT_NAME + YEAR_RANGE + KEYWORDS + DATE_YYYYMMDD

[RUN] Query 1/1: Custom_1
(FT~"test") AND buyer-country IN (SWE) AND publication-date = (20240101 <> 20251231) AND (classification-cpv = 33* OR classification-cpv = 38* OR classification-cpv = 504*)
Page 1: +100 rows (total=100) | [Translate] Detected 73 non-English, 73 successfully translated
Page 2: +100 rows (total=200) | [Translate] Detected 79 non-English, 79 successfully translated
Page 3: +18 rows (tota

[INFO] Drive shortcut path detected; writing XLSX to local temp file: ted_output_ted_consolidated_raw_3598.xlsx
[SAVE] Appended 148 new records to ted_consolidated_raw.xlsx
Consolidated: +148 rows (total=148) -> ted_consolidated_raw.xlsx
[SAVE] Uploaded XLSX to target Google Drive folder URL.
[SAVE] Output directory path: /content/drive/.shortcut-targets-by-id/1aRK0Rb3vPO3OQdyGwTDvvuZCTA7taNBE
[SAVE] Output directory URL:  https://drive.google.com/drive/folders/1aRK0Rb3vPO3OQdyGwTDvvuZCTA7taNBE
[SAVE] Raw output URL for sync: https://docs.google.com/spreadsheets/d/1SG4AW5Bc_Hdiuug9N3EPmxWIKq1e8Vv4/edit?usp=drivesdk&ouid=113770596956228980313&rtpof=true&sd=true
[SAVE] Files successfully generated in: /content/drive/.shortcut-targets-by-id/1aRK0Rb3vPO3OQdyGwTDvvuZCTA7taNBE


In [8]:
# @title
# --- 5. Preview Results ---
import pandas as pd
from IPython.display import display

print(f"Total rows after dedup: {len(df_raw)}")
print("Columns:", list(df_raw.columns))

preview_columns = [
    "Notice publication number",
    "source_url",
    "Publication Date",
    "Buyer name",
    "Country",
    "Value of the tender",
    "Description",
    'Description_en'
]

available_columns = [col for col in preview_columns if col in df_raw.columns]
if available_columns:
    display(df_raw[available_columns].head(20))
else:
    display(df_raw.head(20))


Total rows after dedup: 148
Columns: ['publication-number', 'publication-date', 'classification-cpv', 'main-classification-proc', 'main-classification-lot', 'main-classification-part', 'contract-nature-main-proc', 'contract-nature-main-lot', 'contract-nature', 'place-of-performance', 'place-of-performance-city-proc', 'place-of-performance-country-proc', 'place-of-performance-subdiv-proc', 'place-of-performance-add-proc', 'buyer-name', 'buyer-country', 'winner-name', 'winner-selection-status', 'value-prize-lot', 'value-prize-cur-lot', 'tender-value', 'tender-value-cur', 'result-value-lot', 'result-value-cur-lot', 'estimated-value-proc', 'estimated-value-cur-proc', 'total-value', 'description-proc', 'description-lot', 'description-glo', 'title-proc', 'title-lot', 'tendering-party-name', 'participant-name-lot', 'Notice publication number', 'source_url', 'CPV', 'Currency', 'Main classification', 'Main nature of the contract', 'Place of performance', 'Buyer name', 'Country', 'Winner name', 

,Notice publication number,source_url,Publication Date,Buyer name,Country,Value of the tender,Description,Description_en
0,7730-2024,https://ted.europa.eu/en/notice/-/detail/7730-...,2024-01-05,Region Stockholm - Service administration,SWE,100000000,"RFI avseende Blodprovtagningsartiklar 2025, SL...","RFI regarding Blood sampling articles 2025, SL..."
1,32721-2024,https://ted.europa.eu/en/notice/-/detail/32721...,2024-01-17,Region Stockholm - Service administration,SWE,200000000,Upphandlingen utgör en ramavtalsupphandling oc...,The procurement constitutes a framework contra...
2,35292-2024,https://ted.europa.eu/en/notice/-/detail/35292...,2024-01-18,Kungliga Tekniska högskolan,SWE,160000,KTH is seeking tenders for a Universal mechani...,KTH is seeking tenders for a Universal mechani...
3,53731-2024,https://ted.europa.eu/en/notice/-/detail/53731...,2024-01-26,Uppsala Water and Waste AB,SWE,50000000,Uppsala Vatten och Avfall AB bjuder in leveran...,Uppsala Vatten och Avfall AB invites suppliers...
4,60468-2024,https://ted.europa.eu/en/notice/-/detail/60468...,2024-01-30,The Authority for Community Protection and Pre...,SWE,4125000,MSB har ett nytt avtal med EU inom ramen av Re...,MSB has a new agreement with the EU within the...
5,82829-2024,https://ted.europa.eu/en/notice/-/detail/82829...,2024-02-08,Skurup municipality,SWE,3000000,Ramavtal avseende service för Skurups kommuns ...,Framework agreement regarding service for Skur...
6,90654-2024,https://ted.europa.eu/en/notice/-/detail/90654...,2024-02-13,The State Board of Institutions,SWE,2000000,"Psykologiska tester, screening- och bedömnings...","Psychological tests, screening and assessment ..."
7,93415-2024,https://ted.europa.eu/en/notice/-/detail/93415...,2024-02-14,The supply of goods,SWE,,"Region Uppsala-Varuförsörjningen, inbjuder att...","Region Uppsala-Varuförsjöningen, invites you t..."
8,123403-2024,https://ted.europa.eu/en/notice/-/detail/12340...,2024-02-28,Region Värmland,SWE,9676752,Region Värmland (regionen) genomför denna upph...,Region Värmland (the region) carries out this ...
9,171084-2024,https://ted.europa.eu/en/notice/-/detail/17108...,2024-03-21,Region Stockholm - Service administration,SWE,56000000,"RFI avseende Laboratorieartiklar 2025, SLL410....","RFI regarding Laboratory articles 2025, SLL410..."
